# Lab 7 · RAG experiments: change one thing, watch what breaks

**Time:** 30 min · **Work in teams of 3** · **You will:** take the RAG pipeline from Lab 6 and *mutate* it on purpose — change how many chunks are retrieved (top-k), change how the policy is cut into chunks, and plant a wrong or missing chunk — then measure what each change does.

**Idea:** a RAG system fails in two separate places. *Retrieval* may not bring back the right text; the *model* may not use it properly. Measure retrieval on its own first, then look at the answers.

*How to use this notebook: **Kernel → Restart Kernel and Run All Cells**, read each result, then change the `🔧` values and run those cells again (Shift+Enter). Agree in your team what you expect **before** you run each experiment, and write it down.*

In [ ]:
import sys, os
sys.path.insert(0, next(os.path.abspath(d) for d in ("..", "../..") if os.path.exists(os.path.join(d, "slm_lab.py"))))
from slm_lab import *
import re
import pandas as pd
from sentence_transformers import SentenceTransformer

sections = policy_sections()
embedder = SentenceTransformer(EMBED_ID, device=device())
facts = [it for it in eval_items() if it["kind"] == "fact"]
section_of = {f["id"]: f["section"] for f in load_facts()}
gold_text = {f["id"]: next(s["text"] for s in sections if s["title"] == f["section"]) for f in load_facts() if f["kind"] == "fact"}
print(len(sections), "policy sections,", len(facts), "fact questions to test with")

## 7.0 The pieces (read, then run)
A **chunk** is one piece of the policy that gets its own vector. `make_chunks` cuts the policy in one of two ways:
- **section** — one chunk per numbered policy section (what Lab 6 did);
- **fixed** — a blind cut every `size` characters, ignoring where sections begin and end.

`retrieve` returns the best `k` chunks. `hit` asks the only question that matters for retrieval: *did the right text come back at all?* A chunk counts as a hit when it contains a piece of the section that holds the answer.

In [ ]:
def make_chunks(strategy="section", size=300, overlap=0):
    if strategy == "section":
        return [f"{s['title']}. {s['text']}" for s in sections]
    flat = " ".join(f"{s['title']}. {s['text']}" for s in sections)
    step = max(1, size - overlap)
    return [flat[i:i + size] for i in range(0, len(flat), step) if flat[i:i + size].strip()]


def build_index(chunks):
    return chunks, embedder.encode(chunks, normalize_embeddings=True, convert_to_tensor=True)


def retrieve(question, index, k=3, drop_best=False):
    chunks, vecs = index
    scores = vecs @ embedder.encode(question, normalize_embeddings=True, convert_to_tensor=True)
    order = torch.argsort(scores, descending=True).tolist()
    if drop_best:                     # simulate "the right chunk was never indexed"
        order = order[1:]
    return [(float(scores[i]), chunks[i]) for i in order[:k]]


def hit(item, retrieved):
    gold = gold_text[item["id"]]
    pieces = [gold[i:i + 40] for i in range(0, max(1, len(gold) - 40), 10)]
    return any(p in text for _, text in retrieved for p in pieces)


def hit_rate(index, k=3, **kw):
    return sum(hit(it, retrieve(it["question"], index, k, **kw)) for it in facts) / len(facts)


section_index = build_index(make_chunks("section"))
print("hit rate with Lab 6 settings (sections, top-3):", f"{hit_rate(section_index, 3):.0%}")

### 🔧 Try it 7.1 — Mutation 1: change top-k
With `k = 1` only the single best chunk reaches the model; with larger `k` more text reaches it, which helps recall but also brings more distraction and a longer, slower prompt.

**Predict first:** at which `k` does the hit rate stop improving?

🔧 **Try:** change the list of `ks`, for example `[1, 2, 3, 5, 8]`.

In [ ]:
ks = [1, 2, 3, 5]        # 🔧 try other values
rows = []
for k in ks:
    got = retrieve(facts[0]["question"], section_index, k)
    rows.append({"k": k, "hit rate": f"{hit_rate(section_index, k):.0%}",
                 "characters sent to the model (one question)": sum(len(t) for _, t in got)})
topk_table = pd.DataFrame(rows)
topk_table

In [ ]:
rates = {k: hit_rate(section_index, k) for k in ks}
check(rates[max(ks)] >= rates[min(ks)],
      f"Hit rate goes from {rates[min(ks)]:.0%} at k={min(ks)} to {rates[max(ks)]:.0%} at k={max(ks)}, and the prompt grows with every extra chunk. "
      "Pick the smallest k that gets you close to the maximum.",
      "a larger k should never lower the hit rate - check the list of ks.")

### 🔧 Try it 7.2 — Mutation 2: change the chunking
Same policy, same embedding model, same questions — only the way we cut the text changes. Small blind chunks split a rule from its number; very large chunks mix several topics into one vector.

**Predict first:** will 150-character blind chunks beat whole sections? What about 600?

🔧 **Try:** add your own setting to `settings`, for example `("fixed", 300, 100)` — size 300 with an overlap of 100 characters so a split sentence still appears whole in one chunk.

In [ ]:
settings = [("section", 0, 0), ("fixed", 150, 0), ("fixed", 300, 0), ("fixed", 600, 0), ("fixed", 300, 100)]   # 🔧 (strategy, size, overlap)
rows = []
for strategy, size, overlap in settings:
    idx = build_index(make_chunks(strategy, size, overlap))
    rows.append({"strategy": strategy, "size": size or "-", "overlap": overlap or "-", "chunks": len(idx[0]),
                 "hit@1": f"{hit_rate(idx, 1):.0%}", "hit@3": f"{hit_rate(idx, 3):.0%}"})
chunk_table = pd.DataFrame(rows)
chunk_table

In [ ]:
best = chunk_table.sort_values("hit@3", ascending=False).iloc[0]
check(len(chunk_table) >= 3,
      f"Best setting here: {best['strategy']} (size {best['size']}, overlap {best['overlap']}) at {best['hit@3']} hit@3. "
      "There is no universally right chunk size - you measure it on your own documents.",
      "run the settings cell first.")

### 👀 See it 7.3 — Mutation 3: wrong chunks and a missing chunk
Real knowledge bases contain **old versions** nobody deleted, and **gaps**. We plant them and ask the model about paternity leave (the real answer is **20 days**):
- **normal** — right chunk retrieved;
- **wrong** — an outdated chunk saying *10 days* is returned first, but the current chunk is also there;
- **missing** — the paternity chunk is removed, as if it was never indexed;
- **stale only** — the old version is **all there is**: the current chunk was never re-indexed.

**Predict first:** a stronger model may cope when both versions are in front of it. What happens when only the old one is?

In [ ]:
# The 0.5B model from the earlier labs is too small to follow retrieved text reliably, which would muddy this experiment.
# The 3B model of the same family (about 6 GB, the one used in Lab 6) follows the context and admits what it does not know.
tok, model = load("Qwen/Qwen2.5-3B-Instruct")
STALE = "4.3 Paternity Leave (old version 2.1). Employees are entitled to 10 days of paternity leave for the birth or adoption of a child."
RAG_SYSTEM = ("You are the HR assistant of ABC Company. Answer the question in one sentence using ONLY the "
              "policy excerpts below, and cite the section number.\n"
              f"If the excerpts do not contain the answer, reply exactly: {UNKNOWN_REPLY}\n\nPolicy excerpts:\n")


def answer(question, k=3, mode="normal", index=section_index):
    got = retrieve(question, index, k, drop_best=(mode in ("missing", "stale_only")))
    if mode == "wrong":
        got = [(1.0, STALE)] + got[:k - 1]
    if mode == "stale_only":                     # the right chunk is gone, and the old version takes its place
        got = [(1.0, STALE)] + got[:k - 1]
    system = RAG_SYSTEM + "\n\n".join(text for _, text in got)
    reply = chat(model, tok, [[{"role": "system", "content": system}, {"role": "user", "content": question}]], max_new_tokens=80)[0]
    return reply, got


question = "How many days of paternity leave do I get?"
for mode in ("normal", "wrong", "missing", "stale_only"):
    reply, got = answer(question, 3, mode)
    print(f"{mode:8} → {reply}")
    print(f"          retrieved: {[text[:28] for _, text in got]}\n")

In [ ]:
replies = {m: answer(question, 3, m) for m in ("normal", "wrong", "missing", "stale_only")}
check("4.3 Paternity Leave." in replies["normal"][1][0][1]
      and "old version" in replies["wrong"][1][0][1]
      and not any("4.3 Paternity Leave." in t for _, t in replies["missing"][1])
      and "old version" in replies["stale_only"][1][0][1] and not any("4.3 Paternity Leave." in t for _, t in replies["stale_only"][1]),
      "The four cases did what they say: normal found the paternity chunk, 'wrong' put the stale chunk first with the current one still present, 'missing' removed the right text, "
      "'stale only' left the old version as the only one. Compare the answers above: a model can only repeat what it is shown. A stronger model often picks the right chunk when both versions are present, "
      "but when the old version is all there is, it gives a confident wrong answer that even cites a section.",
      "the retrieval did not behave as described - re-run the cells from the top.")
for mode, (reply, _) in replies.items():
    print(f"{mode:8} says 20 days: {'20' in reply}   says 10 days: {'10' in reply}")

### 🔧 Try it 7.4 — Your team's experiment
Pick **one** question, **one** mutation and **one** prediction, run it, and write one sentence on what you found. Ideas:
- a question that is *not* in the policy (*"Is there a free gym?"*) with `k = 5` — does a bigger `k` make the model invent something?
- the `sick leave` question with a stale chunk of your own (edit `STALE`).
- a question phrased with no keyword from the policy (*"My wife is having a baby — how long can I stay home?"*).

In [ ]:
my_question = "My wife is having a baby - how long can I stay home?"   # 🔧
my_k, my_mode = 3, "normal"                                           # 🔧 mode: normal | wrong | missing | stale_only
reply, got = answer(my_question, my_k, my_mode)
print(reply)
print("retrieved:", [f"{s:.2f} {t[:30]}" for s, t in got])

## 🧠 Check your understanding
1. Raising `k` from 3 to 8 improved the hit rate a little but the prompt got much longer. When is the longer prompt *not* worth it?
2. Why can a RAG system answer **confidently and wrongly** even though the model is behaving correctly?
3. Name two cheap things you could add to catch a stale or missing chunk before the answer reaches an employee.

*✍️ Your answer:* 

**Next:** Lab 8 — guardrails: rules that check the question before and the answer after the model.